# Waste YOLOv8 — use the trained model or train again

The supplied `models/best.pt` and `models/best.onnx` are trained three-class **image classifiers** (metal, other, plastic). They do not draw object boxes. See the README for measured results and limits.

For optional full fine-tuning, choose a GPU runtime. This notebook has not itself been executed in Colab; the shipped CPU training, prediction and export scripts were exercised in the build environment.

## 1. Upload and unpack the model package
Upload `Waste_YOLOv8_Trained.zip`. The prepared dataset is only needed if you want to train or evaluate again.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
files.upload()
package = Path("/content/Waste_YOLOv8_Trained.zip")
assert package.is_file(), "Upload Waste_YOLOv8_Trained.zip first"
with zipfile.ZipFile(package) as z:
    root = Path("/content").resolve()
    for member in z.infolist():
        target = (root / member.filename).resolve()
        if not target.is_relative_to(root):
            raise ValueError("Unsafe archive path")
    z.extractall(root)
os.chdir("/content/Waste_YOLOv8")

## 2. Install lightweight inference dependencies

In [ ]:
%pip install -q -r requirements-inference.txt

## 3. Predict a photo
Upload a photo with one item in view. `uncertain` means the confidence or margin check failed; it is not a fourth learned class.

In [ ]:
import sys
sys.path.insert(0, str(Path("scripts").resolve()))
from predict import WasteClassifier
from PIL import Image
classifier = WasteClassifier("models/best.onnx")
uploaded = files.upload()
for filename in uploaded:
    with Image.open(filename) as image:
        display(image)
        print(classifier.predict(image))

## 4. Optional: upload the prepared dataset
Upload `Waste_YOLOv8_Prepared_Data.zip`, or copy it from your mounted Google Drive into `/content`. It contains train/val/test folders plus the exact preparation manifest.

In [ ]:
files.upload()
data_archive = Path("/content/Waste_YOLOv8_Prepared_Data.zip")
if not data_archive.exists():
    data_archive = Path("Waste_YOLOv8_Prepared_Data.zip").resolve()
assert data_archive.is_file()
with zipfile.ZipFile(data_archive) as z:
    destination = Path("/content/Waste_YOLOv8").resolve()
    for member in z.infolist():
        if not (destination / member.filename).resolve().is_relative_to(destination):
            raise ValueError("Unsafe archive path")
    z.extractall(destination)
assert Path("data/manifest.jsonl").is_file()

## 5. Optional: train
The first command reproduces the shipped frozen-backbone training method. The second performs end-to-end fine-tuning. Run only the method you want. On Colab, retain its matched PyTorch/torchvision pair unless reproducing the exact CPU environment.

In [ ]:
%pip install -q ultralytics==8.3.253 onnx==1.19.0 onnxruntime==1.23.2
import torch
print("GPU available:", torch.cuda.is_available())

In [ ]:
# Reproduce the CPU-friendly transfer model in a new output directory.
# !python scripts/train.py --data data --out retrained --weights yolov8n-cls.pt

# Optional full fine-tuning; may take a substantial time on CPU.
!python scripts/finetune.py --data data --weights models/best.pt --out finetuned --epochs 60

## 6. Evaluate the new frozen choice and export
Do not repeatedly tune against test results. For later experiments reserve a new camera test set. These outputs belong to the new model, not the shipped baseline.

In [ ]:
!python scripts/evaluate.py --data data --weights finetuned/models/best.pt --out finetuned/reports
import subprocess
verification = sorted(Path("data/val").rglob("*.jpg"))[:8]
subprocess.run([sys.executable, "scripts/export_onnx.py", "--weights", "finetuned/models/best.pt", "--report", "finetuned/reports/export_verification.json", "--images", *map(str, verification)], check=True)
import shutil
shutil.make_archive("Waste_YOLOv8_Finetuned", "zip", "finetuned")
files.download("Waste_YOLOv8_Finetuned.zip")